In [9]:
import pandas as pd 

orders=pd.read_csv('orders DA.csv')
orders_items=pd.read_csv('order_items DA.csv')
customers=pd.read_csv('customers DA.csv')
inventory=pd.read_csv('inventory DA.csv')
payments=pd.read_csv('payments DA.csv')
products=pd.read_csv('products DA.csv')
returns=pd.read_csv('returns DA.csv')
suppliers=pd.read_csv('suppliers DA.csv')

In [7]:
%pip install pandas matplotlib seaborn openpyxl numpy

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.8 MB 2.9 MB/s eta 0:00:03
   ---- ----------------------------------- 1.0/9.8 MB 2.9 MB/s eta 0:00:03
   ---- ----------------------------------- 1.0/9.8 MB 2.9 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.8 MB 1.4 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.8 MB 1.4 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.8 MB 1.4 MB/s eta 0:00:06
   ---


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
print("orders Table Info:")
print(orders.info())

print("\n Missing Values in Orders:")
print(orders.isnull().sum())

print(orders.duplicated(subset=['order_id']).sum())

orders Table Info:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          50000 non-null  str    
 1   customer_id       50000 non-null  str    
 2   order_date        50000 non-null  str    
 3   order_time        50000 non-null  str    
 4   status            50000 non-null  str    
 5   city              50000 non-null  str    
 6   state             50000 non-null  str    
 7   pincode           50000 non-null  int64  
 8   total_amount      50000 non-null  float64
 9   gst_amount        50000 non-null  float64
 10  shipping_charge   50000 non-null  int64  
 11  discount_amount   50000 non-null  float64
 12  final_amount      50000 non-null  float64
 13  payment_method    50000 non-null  str    
 14  shipping_partner  50000 non-null  str    
 15  tracking_id       50000 non-null  str    
 16  delivered_date    32499 non-null

In [12]:
orders['order_date']=pd.to_datetime(orders['order_date'],format='%d-%m-%Y')
orders['delivered_date']=pd.to_datetime(orders['delivered_date'],format='%d-%m-%Y')

print("Date conversion successfully!\n")

outliers=orders[orders['final_amount']>500000]
print(f"Number of orders above ₹5 lakh: {len(outliers)}")

invalid_dates = orders[orders['delivered_date'] < orders['order_date']]
print(f"Number of orders delivered before they were placed: {len(invalid_dates)}")

Date conversion successfully!

Number of orders above ₹5 lakh: 312
Number of orders delivered before they were placed: 0


In [13]:
payment_counts=payments['status'].value_counts()
print("Payments Status:\n",payment_counts)

failure_rate=(payment_counts.get('Failed',0)/payment_counts.sum())*100
print(f"\nFailure Rate:{failure_rate:2f}%")

invalid_returns=returns[~returns['order_id'].isin(orders['order_id'])]
print(f"\nReturns without matching order ID:{len(invalid_returns)}")

Payments Status:
 status
Success    48252
Failed      1748
Name: count, dtype: int64

Failure Rate:3.496000%

Returns without matching order ID:0


In [14]:
out_of_stock = inventory[inventory['status'] == 'Out of Stock']
print(f"Out of Stock products: {len(out_of_stock)}")

zero_orders = customers[customers['total_orders'] == 0]
print(f"Customers with zero orders: {len(zero_orders)}")

Out of Stock products: 2
Customers with zero orders: 457


# Phase 1: Data Quality Report

**1. Data Volume & Structure**
* The central `orders` table contains 50,000 records across 19 columns. 

**2. Missing Data & Duplicates**
* There are 17,501 missing values in the `delivered_date` column, which logically corresponds to orders that are cancelled, returned, or currently in transit.
* The dataset is perfectly clean regarding duplicate order IDs; exactly 0 were found.

**3. Outliers & Logical Integrity**
* There are 312 extreme outliers with a `final_amount` exceeding ₹5 Lakh. 
* Zero orders have a delivery date prior to the order date, indicating a reliable timestamp tracking system.

**4. Payments & Returns**
* The payment failure rate is approximately 3.5%.
* Referential integrity for returns is perfect; 0 returns are missing an original matching order ID.

**5. Inventory & Customers**
* Only 2 products are currently in "Out of Stock" status.
* There are 457 registered customers who have a total order count of 0 (registered but have not yet purchased).